# GBM WSI QC calibration

Open this notebook from `notebooks/` (or the repository root) using the patching environment; its
code is in `notebooks/review_helper.py` (`rh`), and the cells hold the settings. It reads existing
tiling output and previews the policy on one cohort's patients (`COHORT`, TCGA by default; TCGA
patients from any fold). By default one policy serves every fold and both cohorts, so freeze it before
any training. It does not re-tile slides or show MGMT labels. Every patient whose tiles you view goes
into `calibration_patients`. For a CPTAC policy (`sources: [CPTAC]`), set `COHORT = "CPTAC"` and view
only the few patients reserved for calibration: export.py withholds every CPTAC calibration patient
from the external test.

Change the policy YAML to test other QC settings. Review a seeded sample of accepted, review,
and rejected patches, then look around the candidate blur boundaries. In pale and low-texture
patches, check for readable nuclei and tissue architecture at the original tile scale.

In [1]:
from pathlib import Path

import review_helper as rh

ROOT = Path.home() / "Coding/WSI/data"
SUBSET = "B"           # A, B or full
POLICY = rh.REPO / "configs/qc_policy_cptac.yaml" # rh.REPO / "configs/qc_policy.yaml" for TCGA or rh.REPO / "configs/qc_policy_cptac.yaml" for CPTAC
COHORT = "CPTAC"        # TCGA, or CPTAC for a CPTAC policy (its patients you view are withheld from the test)

s = rh.open_session(ROOT, SUBSET, POLICY, COHORT)
patients = s.patients
print(f"{len(patients)} {COHORT} patients, {len(s.slides)} slides")
if COHORT == "CPTAC":
    print("CPTAC: view only the patients reserved for calibration; each one listed is withheld from the test.")
print(patients)

flagged = rh.flagged_slides(s)
if flagged is None:
    print(f"\nNo slides_{SUBSET}.csv for {s.policy['version']}: run qc.py to list flagged slides.")
else:
    print(f"\n{len(flagged)} flagged {COHORT} slides need a sign-off decision (set SLIDE below to view one):")
    display(flagged)

42 CPTAC patients, 97 slides
CPTAC: view only the patients reserved for calibration; each one listed is withheld from the test.
['C3L-00365', 'C3L-00674', 'C3L-01040', 'C3L-01043', 'C3L-01149', 'C3L-01154', 'C3L-01155', 'C3L-01157', 'C3L-02465', 'C3L-02504', 'C3L-02955', 'C3L-02970', 'C3L-03260', 'C3L-03392', 'C3L-03728', 'C3L-03744', 'C3L-03748', 'C3L-03968', 'C3N-00665', 'C3N-01192', 'C3N-01334', 'C3N-01366', 'C3N-01369', 'C3N-01515', 'C3N-01518', 'C3N-01798', 'C3N-01816', 'C3N-01818', 'C3N-01851', 'C3N-02185', 'C3N-02186', 'C3N-02255', 'C3N-02256', 'C3N-02769', 'C3N-02782', 'C3N-02784', 'C3N-02786', 'C3N-02788', 'C3N-03182', 'C3N-03183', 'C3N-03184', 'C3N-03473']

2 flagged CPTAC slides need a sign-off decision (set SLIDE below to view one):


,patient_id,slide_id,slide_flags,tissue_accept,tissue_review,tissue_reject,edge_accept,edge_review,edge_reject,median_mean_saturation
0,C3N-01851,C3N-01851-21,pale,2997,0,31,91,681,1830,22.13
1,C3N-02255,C3N-02255-22,pale;mask_floor,2643,21,10,45,321,268,16.61


## GrandQC pen slides

Slides whose GrandQC pen fraction is above the policy's `pen_slide_above`; on these, tiles with more
than `pen_tile_review_above` pen go to review (`gq_pen`). The overview shades each pen slide's
thumbnail: `PEN_REVIEW_COLOR` (orange) for tiles sent to review for pen, `PEN_ACCEPTED_COLOR` (cyan)
for accepted tiles where GrandQC still found pen (at least 0.01 of the tile). Check that orange sits
on real ink, and look for cyan on ink. To see the tiles, set `SLIDE` to a slide ID below and `AUTOMATIC_REASON = "gq_pen"` (or
`DECISION = "accept"` for the yellow ones). The patients shown here go into `calibration_patients`.
With the pen rule off (the CPTAC policy), there are no pen slides.

In [2]:
SHOW_PEN_OVERVIEW = True
PEN_REVIEW_COLOR = "orange"       # tiles sent to review for pen (any matplotlib color)
PEN_ACCEPTED_COLOR = "cyan"       # accepted tiles where GrandQC still found pen

pen = rh.pen_slides(s)
if pen is None:
    print(f"No slides_{SUBSET}.csv for {s.policy['version']}: run qc.py first.")
elif pen.empty:
    print("No pen slides (or the policy's pen rule is off).")
else:
    print(f"{len(pen)} {COHORT} pen slides; patients shown (calibration_patients):",
          sorted(pen["patient_id"].unique()))
    display(pen)
    if SHOW_PEN_OVERVIEW:
        rh.show_pen_overview(s, pen, review_color=PEN_REVIEW_COLOR, accepted_color=PEN_ACCEPTED_COLOR)
missing = rh.grandqc_missing(s)
if missing is not None and len(missing):
    print(f"{len(missing)} slides without current GrandQC data (export blocks their patients):")
    display(missing)

No pen slides (or the policy's pen rule is off).


## Tile counts per patient and slide

Counts from qc.py's `slides_<subset>.csv` for this cohort: tissue and edge tiles and their QC
decisions (before human review), `all_tiles` (every tile scored; bare glass is never a tile) and the
tissue accept percentage. `est_export` estimates what export keeps: accepted tissue + edge tiles up
to the per-slide cap, 0 for a slide excluded at sign-off (single-tile decisions and region boxes are
not counted). Patients and slides with the fewest are listed first: slides below the cap
(`below_cap`, `slides_below_cap`) are where reviewing tiles in review can add data. `flags` shows
sign-off flags and `pen` for GrandQC pen slides; `signed_off` comes from review.py's records.
Set `STATS_PATIENT` to a patient ID to list only its slides.

In [3]:
STATS_PATIENT = None              # or a patient ID: show only that patient's slides

per_patient = rh.patient_stats(s)
per_slide = rh.slide_stats(s)
if per_patient is None:
    print(f"No slides_{SUBSET}.csv for {s.policy['version']}: run qc.py first.")
else:
    totals = per_slide[rh.STAT_COUNTS + ["est_export"]].sum()
    print(f"{COHORT}: {len(per_patient)} patients, {len(per_slide)} slides | tissue tiles {totals['tissue_tiles']:,} "
          f"(accept {totals['tissue_accept']:,}, review {totals['tissue_review']:,}, reject {totals['tissue_reject']:,}) "
          f"| edge tiles {totals['edge_tiles']:,} | estimated export {totals['est_export']:,} "
          f"| slides below the cap {int(per_slide['below_cap'].sum())}")
    if STATS_PATIENT is None:
        display(per_patient)
    else:
        display(per_slide[per_slide["patient_id"] == STATS_PATIENT])

CPTAC: 42 patients, 97 slides | tissue tiles 374,507 (accept 369,569, review 1,698, reject 3,240) | edge tiles 75,222 | estimated export 249,885 | slides below the cap 82


,patient_id,slides,tissue_tiles,tissue_accept,tissue_review,tissue_reject,edge_tiles,edge_accept,edge_review,edge_reject,all_tiles,accept_pct,est_export,slides_below_cap,flags,signed_off
0,C3L-03728,1,413,413,0,0,152,20,81,51,565,100.0,433,1,,
1,C3L-03260,1,534,534,0,0,125,14,77,34,659,100.0,548,1,,
2,C3L-02504,2,594,568,11,15,188,12,92,84,782,95.6,580,2,,
3,C3L-02955,1,725,724,0,1,180,9,122,49,905,99.9,733,1,,
4,C3N-02782,1,1031,1028,0,3,534,75,372,87,1565,99.7,1103,1,,
5,C3L-00365,1,1203,1202,0,1,153,21,94,38,1356,99.9,1223,1,,
6,C3L-01155,2,1553,1553,0,0,355,23,154,178,1908,100.0,1576,2,,
7,C3N-03473,3,1724,1724,0,0,296,22,158,116,2020,100.0,1746,3,,
8,C3N-03183,3,1941,1941,0,0,617,85,438,94,2558,100.0,2026,3,,
9,C3N-01366,2,2038,2020,0,18,1989,29,392,1568,4027,99.1,2049,2,,


## Inspect patches

Pick a patient (or a slide), then set a decision, an automatic reason, a confirmed defect,
or a blur band. Leave a filter as `None` to include all values. Every matching tile is put in a
seeded random order; use ◀ Previous / Next ▶ above the montage to page through them, `N` at a
time (`PAGE` sets the starting page; the same seed gives the same order). `COLUMNS` and `TILE_SIZE`
set the montage layout; small tiles get two-line titles. Montage numbers restart
at 0 on each page, and the record cells below act on the page on show. Without ipywidgets
(`pip install ipywidgets`), change `PAGE` and re-run instead. The same score can represent technical
blur or sharp but smooth tissue. A score near a proposed cutoff is a reason to inspect the image.

Cached patches are checked against their original pixel hashes. Uncached patches are
reconstructed from the WSI and checked against the same hash. A missing WSI or unreadable
patch is reported on its panel.

`SLIDE` (the start of a slide ID, e.g. from the flagged list) shows one slide and overrides `PATIENT`.
`ROLE` limits the sample to tissue or edge tiles: the flagged list's review counts are split the same
way, and a slide's edge tiles in review (those missing an `edge_accept_*` cutoff) can far outnumber
its tissue tiles in review.
`PATIENT = None` pages through every TCGA patient in the subset; it reads every slide, so it is slow,
and past 50,000 matches it pages a seeded random 50,000. Every patient listed under "Patients
viewed" (on any page shown) goes into `calibration_patients`. Each title shows the slide,
the tile's level-0 position, and whether its pixels were cached or re-read from the WSI; the GrandQC
line lists only fractions of at least 0.01. With `SHOW_THUMBNAILS`, each sampled slide's QC
thumbnail (green: tissue mask, blue: some stored tiles) is shown with the sampled tiles numbered in
orange; use it to find a tile's place, or a box for `review.append_region`.

In [4]:
PATIENT = "C3N-01851"            # or paste an ID from the list above; None: every patient (TCGA only)
SLIDE = None                      # or the start of a slide ID; overrides PATIENT
ROLE = "tissue"                   # tissue, edge, or None for both
DECISION = None                  # accept, review, reject, or None
AUTOMATIC_REASON = None           # e.g. blur, green_ink, edge, edge_accepted, gq_oof, gq_pen, invalid_score
CONFIRMED_DEFECT = None           # e.g. blur, fold, pen, or None
BLUR_BAND = None                # e.g. (50, 100), lower inclusive, upper exclusive
N = 12                            # tiles per page
COLUMNS = 6                       # tiles per row
TILE_SIZE = 2.2                   # inches per tile; below 3 the titles are shortened (the table has the rest)
PAGE = 0                          # page to start on (0 = first)
SEED = 42                         # order of the tiles; change it for another order
SHOW_THUMBNAILS = True

slides = rh.select_slides(s, PATIENT, SLIDE)
tiles, matches = rh.match_tiles(s, slides, decision=DECISION, reason=AUTOMATIC_REASON,
                                confirmed_defect=CONFIRMED_DEFECT, blur_band=BLUR_BAND, role=ROLE, seed=SEED)
print(f"{matches:,} matches" + (f"; paging a random {len(tiles):,} of them" if len(tiles) < matches else ""))
pager = rh.TilePager(s, tiles, per_page=N, page=PAGE, thumbnails=SHOW_THUMBNAILS, columns=COLUMNS,
                     tile_size=TILE_SIZE)
pager.show()

3,028 matches


## Record a calibration observation

Choose a row number from the montage page on show. Record whether the patch is usable, unusable, or
uncertain, and enter a defect only if you confirmed it. Use `none` when no defect is confirmed.
This journal is scoped to the fold map and policy preview. It is a calibration record.
Decisions that change the export (patch decisions and slide sign-off) go in the next cell.

In [ ]:
SAVE = False
ROW_NUMBER = 0
USABLE = "uncertain"             # usable, unusable or uncertain
CONFIRMED = "none"               # allowed values below
REVIEWER = ""
NOTES = ""

print("Confirmed defect choices:", ", ".join(sorted(rh.review.DEFECTS)))
if SAVE:
    rh.save_observation(s, pager.current(), ROW_NUMBER, USABLE, CONFIRMED, REVIEWER, NOTES)
summary = rh.observation_summary(s)
if summary is not None:
    reviewed, counts = summary
    print("Reviewed patient IDs:", reviewed)
    display(counts)

## Record production decisions

These go to `data_prep/review.py`'s CSVs under `<root>/review/<tiling version>/`, which export.py
applies on top of the QC decisions; unlike the calibration journal above, they change the export.
Records are only appended, and the latest record per tile and per slide wins. Choose montage numbers
on the page on show in `ROWS` to accept (reinstate), reject or send back to review, with a defect type. To sign off a
slide, show only that slide (set `SLIDE`) and set `SIGN_OFF` to approve, or exclude with a reason
in `NOTES`. For a box around an artifact, call `review.append_region` (see the README).

In [ ]:
SAVE_DECISIONS = False
ROWS = []                         # montage numbers on the page on show, e.g. [3, 7]
RECORD = "reject"                 # accept, reject or review
DEFECT = "none"                   # a defect choice listed by the cell above
SIGN_OFF = None                   # approve or exclude the one slide in the montage, or None
SIGN_OFF_CHECKS = {}              # what you checked, e.g. {"pale": True, "focus": True}
REVIEWER = ""
NOTES = ""                        # required to exclude a slide

print("Sign-off checks:", ", ".join(rh.review.CHECKS))
if SAVE_DECISIONS:
    rh.record_decisions(s, pager.current(), ROWS, RECORD, DEFECT, SIGN_OFF, SIGN_OFF_CHECKS, REVIEWER, NOTES)
rh.decision_status(s, pager.current());

## Compare proposed blur cutoffs

Each pair is `(reject below, review below)`; a review cutoff of `None` means no review band.
Other rules stay fixed. Inspect the accepted,
review and rejected counts per slide and patient. `accepted_cached` shows how many of
the accepted tiles have pixels already in the cache. These are automatic QC counts
before any human override. A sweep over every `COHORT` patient in the subset reads all its tissue scores.

Choose cutoffs after examining reviewer decisions and per-patient retention, including
pale and low-texture tissue. Set `calibration_patients`, `reviewer` and `notes` in a new
policy YAML. Record all patients whose slides or retention results informed the choice.
Freeze the policy after the team approves the evidence, before any training. Use that same
policy for A and B and every fold, and for CPTAC too unless the team gives CPTAC its own policy.

In [ ]:
SWEEP_PATIENTS = pager.viewed_patients()   # patients on the pages shown; patients for every one
CANDIDATES = [(50, 100), (100, None), (150, None)]

slide_counts, patient_counts = rh.sweep_blur(s, SWEEP_PATIENTS, CANDIDATES)
if slide_counts is None:
    print("No slides selected.")
else:
    display(slide_counts)
    display(patient_counts)